In [5]:
import pandas as pd
import numpy as np
import json
import xgboost as xgb
from datetime import datetime
from calendar import month_name

In [18]:
DATA_DIR = 'Rental_Predictor_Data'
MONTH_NAMES = [month_name[i] for i in range(1, 13)]

In [22]:
class RentForecaster:
 
    def __init__(self, data_dir=DATA_DIR):
        print("Loading RentForecaster")
        self.data_dir = data_dir
        self._load_artifacts()
        print("Ready.\n")
 
    def _load_artifacts(self):
        # Model
        with open(f'{self.data_dir}/model_features.json') as f:
            self.meta = json.load(f)
 
        self.model = xgb.XGBRegressor()
        self.model.load_model(f'{self.data_dir}/model_xgb.json')
        self.features      = self.meta['features']
        self.encoding_maps = self.meta['encoding_maps']
        self.clip_upper    = self.meta['clip_upper']
        self.clip_lower    = self.meta['clip_lower']
 
        # Reference data
        self.tier_map = pd.read_csv(f'{self.data_dir}/zip_tier_map.csv',
                                    dtype={'zip_code': str})
        self.tier_map['zip_code'] = self.tier_map['zip_code'].str.zfill(5)
 
        self.seasonal = pd.read_csv(f'{self.data_dir}/features_seasonal.csv',
                                    dtype={'zip_code': str})
        self.seasonal['zip_code'] = self.seasonal['zip_code'].str.zfill(5)
 
        self.metro_seasonal = pd.read_csv(f'{self.data_dir}/features_metro_seasonal.csv')
 
        # Historical data for feature engineering
        self.history = pd.read_csv(f'{self.data_dir}/zori_safmr_joined.csv',
                                   low_memory=False,
                                   parse_dates=['date'])
        self.history['zip_code'] = self.history['zip_code'].astype(str).str.zfill(5)
        self.history = self.history.sort_values(['zip_code', 'date'])
 
        print(f"  Model features:  {len(self.features)}")
        print(f"  ZIPs in tier map: {len(self.tier_map):,}")
        print(f"  History rows:     {len(self.history):,}")
 
    def predict(self, zip_code: str, bedrooms: int) -> dict:
        # Predictys 12 month rent forcast and seasonality for a zip code & bedroom count
        
        zip_code = str(zip_code).zfill(5)
 
        if bedrooms not in range(5):
            raise ValueError(f"bedrooms must be 0-4, got {bedrooms}")
 
        tier_row = self.tier_map[self.tier_map['zip_code'] == zip_code]
 
        if len(tier_row) == 0:
            return self._not_found(zip_code)
 
        tier  = int(tier_row['tier'].iloc[0])
        state = str(tier_row['state'].iloc[0])
        metro = str(tier_row['metro'].iloc[0])
        city  = str(tier_row['city'].iloc[0])
 
        zip_hist = self.history[self.history['zip_code'] == zip_code].copy()
        zip_hist_zori = zip_hist.dropna(subset=['zori'])
 
        if len(zip_hist_zori) == 0:
            return self._not_found(zip_code)
 
        # Most recent row with ZORI data
        latest = zip_hist_zori.sort_values('date').iloc[-1]
        current_zori  = float(latest['zori'])
        current_date  = latest['date']
        fiscal_year   = int(latest['fiscal_year'])
 
        br_col     = f'fmr_{bedrooms}br'
        ratio_col  = f'ratio_{bedrooms}br_to_2br'
 
        fmr_value  = latest.get(br_col, np.nan)
        ratio      = latest.get(ratio_col, np.nan)
 
        # Fallback: compute ratio from latest available SAFMR data
        if pd.isna(ratio) or pd.isna(fmr_value):
            safmr_rows = zip_hist.dropna(subset=[br_col, 'fmr_2br'])
            if len(safmr_rows) > 0:
                last_safmr = safmr_rows.sort_values('date').iloc[-1]
                fmr_value  = float(last_safmr[br_col])
                ratio      = float(last_safmr[br_col]) / float(last_safmr['fmr_2br'])
            else:
                ratio     = 1.0
                fmr_value = current_zori
 
        # Current bedroom-specific rent estimate
        current_br_rent = current_zori * ratio
 
        features_dict = self._build_features(
            zip_hist_zori, zip_hist, latest, state, metro, current_date
        )
 
        forecast_yoy_pct  = None
        forecast_rent_12m = None
        warning           = None
 
        if tier == 1 and features_dict is not None:
            X = pd.DataFrame([features_dict])[self.features]
            pred = float(self.model.predict(X.to_numpy())[0])
            pred = np.clip(pred, self.clip_lower, self.clip_upper)
 
            forecast_yoy_pct  = round(pred, 2)
            forecast_rent_12m = round(current_br_rent * (1 + pred / 100))
 
        elif tier == 2:
            warning = (
                "Limited ZORI history for this zip code. "
                "Rent forecast unavailable, seasonality profile only."
            )
        else:
            warning = (
                f"Insufficient data for ZIP {zip_code}. "
                f"Showing metro-level seasonality for {metro}."
            )
 
        seasonality = self._get_seasonality(zip_code, tier, metro, current_br_rent)
 
        # Best and worst months
        best_month  = seasonality[np.argmin([s['avg_deviation_pct'] for s in seasonality])]['month']
        worst_month = seasonality[np.argmax([s['avg_deviation_pct'] for s in seasonality])]['month']
        return {
            'zip_code':          zip_code,
            'city':              city,
            'state':             state,
            'metro':             metro,
            'tier':              tier,
            'bedroom_count':     bedrooms,
            'as_of_date':        current_date.strftime('%Y-%m'),
            'current_est_rent':  round(current_br_rent),
            'forecast_yoy_pct':  forecast_yoy_pct,
            'forecast_rent_12m': forecast_rent_12m,
            'best_month':        best_month,
            'worst_month':       worst_month,
            'seasonality':       seasonality,
            'data_warning':      warning,
        }
 
    def _build_features(self, zip_hist_zori, zip_hist, latest, state, metro, current_date):
        # Engineer the same features used during training from zip code history.
 
        zori_series = zip_hist_zori.set_index('date')['zori'].sort_index()
 
        if len(zori_series) < 36:
            return None
 
        # Lag features
        def lag(n):
            target_date = current_date - pd.DateOffset(months=n)
            # Find closest available date within 45 days
            diffs = abs(zori_series.index - target_date)
            if diffs.min().days > 45:
                return np.nan
            return float(zori_series.iloc[diffs.argmin()])
 
        zori_lag_1m  = lag(1)
        zori_lag_3m  = lag(3)
        zori_lag_6m  = lag(6)
        zori_lag_12m = lag(12)
        zori_lag_24m = lag(24)
        zori_lag_36m = lag(36)
 
        current_zori = float(latest['zori'])
 
        # Rate of change
        zori_mom_pct = (current_zori - zori_lag_1m)  / zori_lag_1m  * 100 if zori_lag_1m  else np.nan
        zori_yoy_pct = (current_zori - zori_lag_12m) / zori_lag_12m * 100 if zori_lag_12m else np.nan
        zori_2y_pct  = (current_zori - zori_lag_24m) / zori_lag_24m * 100 if zori_lag_24m else np.nan
 
        # Rolling stats (prior 6 and 12 months)
        recent_6  = zori_series[zori_series.index < current_date].tail(6)
        recent_12 = zori_series[zori_series.index < current_date].tail(12)
        zori_roll_mean_6m  = float(recent_6.mean())  if len(recent_6)  >= 3 else np.nan
        zori_roll_mean_12m = float(recent_12.mean()) if len(recent_12) >= 6 else np.nan
        zori_roll_std_6m   = float(recent_6.std())   if len(recent_6)  >= 3 else np.nan
 
        # SAFMR features
        safmr_rows = zip_hist.dropna(subset=['fmr_1br']).sort_values('date')
        if len(safmr_rows) >= 2:
            last  = safmr_rows.iloc[-1]
            prior = safmr_rows.iloc[-2]
            def safmr_yoy(col):
                curr = last.get(col, np.nan)
                prev = prior.get(col, np.nan)
                if pd.isna(curr) or pd.isna(prev) or prev == 0:
                    return np.nan
                return (curr - prev) / prev * 100
            fmr_0br = float(last.get('fmr_0br', np.nan))
            fmr_1br = float(last.get('fmr_1br', np.nan))
            fmr_2br = float(last.get('fmr_2br', np.nan))
            fmr_0br_lag1y    = float(prior.get('fmr_0br', np.nan))
            fmr_1br_lag1y    = float(prior.get('fmr_1br', np.nan))
            fmr_2br_lag1y    = float(prior.get('fmr_2br', np.nan))
            fmr_0br_yoy_pct  = safmr_yoy('fmr_0br')
            fmr_1br_yoy_pct  = safmr_yoy('fmr_1br')
            fmr_2br_yoy_pct  = safmr_yoy('fmr_2br')
        else:
            fmr_0br = fmr_1br = fmr_2br = np.nan
            fmr_0br_lag1y = fmr_1br_lag1y = fmr_2br_lag1y = np.nan
            fmr_0br_yoy_pct = fmr_1br_yoy_pct = fmr_2br_yoy_pct = np.nan
 
        # Calendar
        month   = current_date.month
        quarter = (month - 1) // 3 + 1
        year    = current_date.year
 
        # Luxury flag
        zip_median = float(zori_series.median())
        is_luxury  = 1 if zip_median >= 2500 else 0
 
        # Categorical encoding
        metro_encoded = self.encoding_maps.get('Metro', {}).get(metro, np.nan)
        state_encoded = self.encoding_maps.get('State', {}).get(state, np.nan)
 
        return {
            'zori_lag_1m':        zori_lag_1m,
            'zori_lag_3m':        zori_lag_3m,
            'zori_lag_6m':        zori_lag_6m,
            'zori_lag_12m':       zori_lag_12m,
            'zori_lag_24m':       zori_lag_24m,
            'zori_lag_36m':       zori_lag_36m,
            'zori_roll_mean_6m':  zori_roll_mean_6m,
            'zori_roll_mean_12m': zori_roll_mean_12m,
            'zori_roll_std_6m':   zori_roll_std_6m,
            'zori_mom_pct':       zori_mom_pct,
            'zori_yoy_pct':       zori_yoy_pct,
            'zori_2y_pct':        zori_2y_pct,
            'fmr_0br':            fmr_0br,
            'fmr_1br':            fmr_1br,
            'fmr_2br':            fmr_2br,
            'fmr_0br_lag1y':      fmr_0br_lag1y,
            'fmr_1br_lag1y':      fmr_1br_lag1y,
            'fmr_2br_lag1y':      fmr_2br_lag1y,
            'fmr_0br_yoy_pct':    fmr_0br_yoy_pct,
            'fmr_1br_yoy_pct':    fmr_1br_yoy_pct,
            'fmr_2br_yoy_pct':    fmr_2br_yoy_pct,
            'month':              month,
            'quarter':            quarter,
            'month_sin':          np.sin(2 * np.pi * month / 12),
            'month_cos':          np.cos(2 * np.pi * month / 12),
            'year':               year,
            'is_luxury':          is_luxury,
            'Metro_encoded':      metro_encoded,
            'State_encoded':      state_encoded,
        }
 
    def _get_seasonality(self, zip_code, tier, metro, current_br_rent):
        # Return monthly seasonal profile, with metro fallback for Tier 3.
 
        if tier in [1, 2]:
            rows = self.seasonal[self.seasonal['zip_code'] == zip_code]
        else:
            rows = self.metro_seasonal[self.metro_seasonal['Metro'] == metro]
            rows = rows.rename(columns={'avg_dev_pct': 'avg_deviation_pct'})
 
        if len(rows) == 0:
            # Final fallback: flat profile
            return [
                {
                    'month':             MONTH_NAMES[i],
                    'month_num':         i + 1,
                    'avg_deviation_pct': 0.0,
                    'est_rent':          round(current_br_rent),
                    'reliable':          False,
                }
                for i in range(12)
            ]
 
        profile = []
        for m in range(1, 13):
            row = rows[rows['month'] == m]
            if len(row) == 0:
                dev = 0.0
                reliable = False
            else:
                dev      = float(row['avg_deviation_pct'].iloc[0] if 'avg_deviation_pct' in row.columns else row['avg_dev_pct'].iloc[0])
                reliable = bool(row['reliable'].iloc[0]) if 'reliable' in row.columns else True
 
            profile.append({
                'month':             MONTH_NAMES[m - 1],
                'month_num':         m,
                'avg_deviation_pct': round(dev, 2),
                'est_rent':          round(current_br_rent * (1 + dev / 100)),
                'reliable':          reliable,
            })
 
        return profile
 
    # Helper Functions
 
    def _not_found(self, zip_code):
        return {
            'zip_code':      zip_code,
            'error':         f"ZIP code {zip_code} not found in dataset.",
            'data_warning':  "This ZIP code has no available data.",
        }
 
    def summary(self, result: dict):
        # Print a human readable summary of a prediction result.
        if 'error' in result:
            print(f"Error: {result['error']}")
            return
 
        print(f"\n{'='*55}")
        print(f"  Rent Forecast: {result['zip_code']} ({result['city']}, {result['state']})")
        print(f"  Metro: {result['metro']}")
        print(f"  Data tier: {result['tier']}  |  As of: {result['as_of_date']}")
        print(f"{'='*55}")
        print(f"  Bedrooms:          {result['bedroom_count']}BR")
        print(f"  Current est. rent: ${result['current_est_rent']:,}/mo")
 
        if result['forecast_yoy_pct'] is not None:
            direction = "up" if result['forecast_yoy_pct'] > 0 else "down"
            print(f"  12mo forecast:     ${result['forecast_rent_12m']:,}/mo "
                  f"({direction} {abs(result['forecast_yoy_pct']):.1f}% YoY)")
        else:
            print(f"  12mo forecast:     Not available")
 
        print(f"\n  Best month to sign: {result['best_month']}")
        print(f"  Most expensive month: {result['worst_month']}")
 
        if result['data_warning']:
            print(f"\n  Note: {result['data_warning']}")
 
        print(f"\n  Monthly Seasonality:")
        print(f"  {'Month':<12} {'Deviation':>10} {'Est. Rent':>12}")
        print(f"  {'-'*36}")
        for s in result['seasonality']:
            marker = ''
            if s['month'] == result['best_month']:
                marker = ' <- cheapest'
            elif s['month'] == result['worst_month']:
                marker = ' <- most expensive'
            rel = '' if s['reliable'] else '*'
            print(f"  {s['month']:<12} {s['avg_deviation_pct']:>+9.1f}%  "
                  f"${s['est_rent']:>8,}{rel}{marker}")
        print(f"\n  * Low confidence months (fewer than 3 observations)")
        print(f"{'='*55}\n")

In [23]:
forecaster = RentForecaster()

# Test Carlsbad
result = forecaster.predict(zip_code='92008', bedrooms=1)
forecaster.summary(result)

# Test a Tier 2 ZIP
result2 = forecaster.predict(zip_code='44116', bedrooms=2)
forecaster.summary(result2)

# Test an invalid ZIP
result3 = forecaster.predict(zip_code='00000', bedrooms=1)
forecaster.summary(result3)

Loading RentForecaster...
  Model features:  29
  ZIPs in tier map: 8,316
  History rows:     1,056,132
Ready.


  Rent Forecast: 92008 (Carlsbad, CA)
  Metro: San Diego-Chula Vista-Carlsbad, CA
  Data tier: 1  |  As of: 2026-04
  Bedrooms:          1BR
  Current est. rent: $2,600/mo
  12mo forecast:     $2,652/mo (up 2.0% YoY)

  Best month to sign: January
  Most expensive month: September

  Monthly Seasonality:
  Month         Deviation    Est. Rent
  ------------------------------------
  January           -3.6%  $   2,507 <- cheapest
  February          -2.8%  $   2,528
  March             -2.0%  $   2,549
  April             -1.1%  $   2,572
  May               -0.5%  $   2,587
  June              +0.5%  $   2,613
  July              +1.3%  $   2,635
  August            +1.8%  $   2,648
  September         +2.2%  $   2,657 <- most expensive
  October           +1.6%  $   2,643
  November          +1.5%  $   2,640
  December          +1.5%  $   2,639

  * Low confidence months (f